# *Libraries*

In [5]:
from typing import TypedDict, Optional

import os

import fitz

import random

import chromadb

from docx import Document

from uuid import uuid4

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI



# *LLM*

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# *Agent State*

In [ ]:
class AgentState(TypedDict):

    # File
    file_path: Optional[str]
    file_name: Optional[str]
    file_type: Optional[str]

    # Validation
    validation_status: Optional[str]
    validation_reason: Optional[str]

    # Sections
    sections: list[dict]

    # Chunks
    chunks : list[dict]

    # Indexing stauts
    indexing_status : str

    # Suggested/Selected Topics or SubTopics
    selected_topic_or_subtopic : Optional[str]
    suggested_topics : Optional[str]

    # Selected Questions type
    selected_questions_type : Optional[str]
    

# *Upload API Node*

In [8]:
MAX_FILE_SIZE = 10

VALID_EXTENSION = [".pdf", ".txt", ".docx"]



In [9]:
def upload_api(state : AgentState, file_path : str) -> AgentState:
    """
    Validates uploaded document before entering RAG pipeline.

    Checks:
    - file existence
    - supported extension
    - file size limit

    Updates workflow state with validation results.
    """

    # File Existence Check
    if not os.path.exists(file_path):

        state["validation_status"] = "failed"
        state["validation_reason"] = "file_not_found"

        return state
    
    # Retreive File Name and Extension
    file_name = os.path.basename(file_path)

    _, extension = os.path.splitext(file_name)

    # File Extensions Check
    if extension.lower() not in VALID_EXTENSION:

        state["validation_status"] = "failed"
        state["validation_reason"] = "unsupported_file_type"

        return state
    
    # File Size Check
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

    if file_size_mb > MAX_FILE_SIZE:

        state["validation_status"] = "failed"
        state["validation_reason"] = "file_too_large"

        return state

    # Success 
    state["file_path"] = file_path
    state["file_name"] = file_name
    state["file_type"] = extension.lower()

    state["validation_status"] = "success"
    state["validation_reason"] = ""

    return state

# *Document Validation Node*

In [10]:
def document_validation_node(state : AgentState) -> AgentState:
    """
    Validates whether uploaded document
    is safe and usable for the RAG pipeline.

    Checks:
    - corrupted PDF
    - encrypted PDF
    - extractable text
    - empty document
    - scanned/image-only PDF
    """

    # Reterive file info from state
    file_path = state["file_path"]
    file_type = state["file_type"]

    # Validates .txt and .docx files
    if file_type  in [".txt", ".docx"]:

        state["validation_status"] = "success"
        
        return state
    
    # Validates PDF's
    # Corrupted pdf check
    try:
        # Open pdf
        doc = fitz.open(file_path)

    except Exception:
        state["validation_status"] = "failed"
        state["validation_reason"] = "corrupted_pdf"

        return state
    
    # Encrypted pdf check
    if doc.is_encrypted():
        state["validation_status"] = "failed"
        state["validation_reason"] = "encrypted_pdf"

        return state
    
    # Extract text
    extracted_text = ""

    for page in doc:
        extracted_text += page.get_text()

    # Empty pdf check
    if len(extracted_text.strip()) == 0:
        state["validation_status"] = "failed"
        state["validation_reason"] = "empty_pdf"

        return state  

    # Scanned/Image only pdf check
    if len(extracted_text.strip()) < 100:
        state["validation_status"] = "failed"
        state["validation_reason"] = "scanned_or_image_only_pdf"

        return state  
    
    # Success
    state["validation_status"] = "success"
    state["validation_reason"] = ""

    doc.close()

    return state    
    

# *Document Ingestion Node*

In [11]:
def document_ingestion_node(state : AgentState) -> AgentState:
    """
    Extract structured readable content 
    from validate document 

    strategy :-
    - for txt : full text extraction
    - for docx : pagragph based extraction
    - for pdf : page based extraction
    """

    # Retriving document info
    file_path = state["file_path"]
    file_type = state["file_type"]

    sections = []

    # txt text extraction
    if file_type == ".txt":

        with open(file_path, "r", encoding="utf-8") as file:
            
            text = file.read()

        sections.append({
            "page" : 1,
            "text" : text.strip()
        })

    # docx text extraction
    elif file_type == ".docx":

        doc = Document(file_path)

        full_text = ""

        for paragraph in doc.paragraphs:
            
            if paragraph.text.strip():

                full_text += paragraph.text + "\n"

        sections.append({
            "page" : 1,
            "text" : full_text.strip()
        })

    # pdf text extraction
    elif file_type == ".pdf":
        doc = fitz.open(file_path)
 
        for page_number, page in enumerate(doc):
            text = page.get_text()

            if len(text.strip()) == 0:
                continue
        
            sections.append({
                "page" : page_number + 1,
                "text" : text.strip()
            })
            
        doc.close()

    # Store Sections in state
    state["sections"] = sections
    
    return state

        

# *Semantic Chunking Node*

In [12]:
def semantic_chunking_node(state : AgentState) -> AgentState:
    """
    Converts extracted documetn setions 
    into semantically meaningful chunks.

    Strategy:
    - Recursive semantic splitting
    - Overlap preservation
    - Page tracking
    """

    # Reterive sections info from state
    sections = state["sections"]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap = 150,
        length_function = len
    )

    chunks = []

    for section in sections:

        page = section["page"]
        text = section["text"]

        # Skip empty text 
        if len(text.strip()) == 0:
            continue

        # Generate semantic chunks
        split_chunks = text_splitter.split_text(text)

        # Store chunks meta data
        for chunk_order, chunk_text in enumerate(split_chunks):
            chunks.append({
                "chunk_id" : str(uuid4()),
                "page" : page,
                "chunk_order" : chunk_order + 1,
                "text" : chunk_text
            })

    state["chunks"] = chunks

    return state
        

# *Indexing Node*

In [13]:
chroma_client = chromadb.PersistentClient(path = "./chroma_db")

collection = chroma_client.get_or_create_collection(name="educational_chunks")

def indexing_node(state : AgentState) -> AgentState:
    """
    Converts chunks into embeddings 
    and store them in chromadb.
    """

    chunks = state["chunks"]
    
    # Process each chunk
    for chunk in chunks:

        # Reterive info from chunks
        chunk_id = chunk["chunk_id"]
        page = chunk["page"]
        text = chunk["text"]
        chunk_order = chunk["chunk_order"]

        # Generate embeddings
        response = llm.embeddings.create(
            model = "text-embedding-3-small",
            input = text
        )

        embeddings = response.data[0].embedding

        # Store in ChromaDB
        collection.add(
            ids = [chunk_id],
            documents = [text],
            embeddings = [embeddings],
            metadatas = [{
                "page" : page,
                "chunk_order" : chunk_order
            }]

        )

    # Store Indexing status
    state["indexing_status"] = "success"

    return state

# *Topic/SubTopic Suggestions and Selection Node*

In [14]:
def topic_suggestion_selection_node(state: AgentState) -> AgentState:
    """
    Generates topic/subtopic names
    from educational material and
    lets user select topic/subtopic.

    Strategies:
    - Random chunk sampling
    - LLM-based topic extraction
    """

    chunks = state["chunks"]

    combined_text = ""

    # Random chunk sampling
    sample_chunks = random.sample(chunks, min(15, len(chunks)))

    for chunk in sample_chunks:

        combined_text += chunk["text"] + "\n\n"

    response = llm.chat.completions.create(

        model="gpt-4o-mini",

        messages=[

            {
                "role": "system",

                "content":
                """
                You are an educational topic extractor.

                Extract the major topics and subtopics
                from the provided educational content.

                Return only clean bullet points.

                Example:

                - Binary Trees
                - Tree Traversal
                - Graph Algorithms
                """
            },

            {
                "role": "user",

                "content": combined_text
            }
        ]
    )

    # Extract generated topics
    suggested_topics = response.choices[0].message.content

    # Display suggested topics
    print("\nSuggested Topics:\n")

    print(suggested_topics)

    # User selection
    selected_topic_or_subtopic = input("Enter topic or subtopic: ")

    # Store in state
    state["suggested_topics"] = suggested_topics

    state["selected_topic_or_subtopic"] = selected_topic_or_subtopic

    return state

# *MCQ's and Theory Type Questions Selections Node*

In [17]:
def select_mcq_or_theory_ques_node(state : AgentState) -> AgentState:
    """
    Take user input for selecting 
    question generation type.
    """

    # Display Questions type
    print("Select questions type:")
    print("1. MCQ Questions")
    print("2. Theory Questions")

    # Input Validation loop
    while True:

        user_input = input("Enter questions type(1 or 2): ")

        if user_input in ["1", "2"]:
            break

        print("Invalid input. Please Enter 1 or 2.")

    # Store selected question type in state
    if user_input == "1":
        state["selected_questions_type"] = "mcq"
    else: 
        state["selected_questions_type"] = "theory"

    return state